In [32]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.service import Service
import pandas as pd
import traceback
import time
import os

# 總計時開始
total_start = time.time()

# Selenium 設定
options = webdriver.ChromeOptions()
# options.add_argument('--headless')  # 若不需顯示瀏覽器，取消註解此行
service = Service(r"C:\Users\anaco\chromedriver-win64\chromedriver-win64\chromedriver.exe")
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 3)

# 網站與欄位對照設定
base_url = "https://pma.gov.taipei/News.aspx?n=6833ECE829BE5990&sms=504AB58CAE8A1C62&page={}&PageSize=200"
field_map = {
    "停車場名稱": "name",
    "費率": "rate",
    "行政區": "district",
    "停車場地址": "addr",
    "機車": "moto",
    "小車": "car",
    "大車": "bus",
    "停車場電話": "phone",
    "開放時間": "open_time"
}
field_order = ["name", "rate", "district", "addr", "moto", "car", "bus", "phone", "open_time"]

# 等待表格穩定（列數不再變動）
def wait_for_table_stable(driver, wait_time=1, max_wait=10):
    prev_count = -1
    elapsed = 0
    while elapsed < max_wait:
        try:
            rows = driver.find_elements(By.CSS_SELECTOR, "#CCMS_Content .area-table.rwd-straight table tbody > tr")
            curr_count = len(rows)
            if curr_count == prev_count:
                return rows
            prev_count = curr_count
        except:
            pass
        time.sleep(wait_time)
        elapsed += wait_time
    return []

# 擷取資料
data = []
page = 1

while page <= 2:
    try:
        page_start = time.time()  # 單頁計時開始

        url = base_url.format(page)
        driver.get(url)
        print(f"擷取第 {page} 頁...")

        table = wait.until(EC.presence_of_element_located(
            (By.CSS_SELECTOR, "#CCMS_Content .area-table.rwd-straight table")
        ))

        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

        rows = wait_for_table_stable(driver)

        for i, row in enumerate(rows, start=1):
            try:
                row_data = {k: "" for k in field_order}
                cols = row.find_elements(By.TAG_NAME, "td")
                for col in cols:
                    key = col.get_attribute("data-title").strip()
                    val = col.text.strip()
                    if key in field_map:
                        row_data[field_map[key]] = val

                full_addr = row_data["district"] + row_data["addr"]
                row_data["地址"] = full_addr
                del row_data["district"]
                del row_data["addr"]

                data.append(row_data)

            except Exception as e:
                print(f"第 {page} 頁第 {i} 列錯誤：{e}")
                traceback.print_exc()

        page_end = time.time()
        print(f"第 {page} 頁完成，用時 {page_end - page_start:.2f} 秒")
        page += 1

    except WebDriverException as we:
        print(f"WebDriver 錯誤：{we}")
        traceback.print_exc()
        break
    except Exception as e:
        print(f"未知錯誤：{e}")
        traceback.print_exc()
        break

driver.quit()
print("所有資料擷取完畢")

# 儲存資料
df = pd.DataFrame(data)
df = df[["name", "rate", "地址", "moto", "car", "bus", "phone", "open_time"]]
df.columns = ["停車場名稱", "費率", "地址", "機車", "小車", "大車", "電話", "開放時間"]


output_csv_path = os.path.join(os.getcwd(), "台北市停車場.csv")
df.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print(f"CSV 已儲存至：{output_csv_path}")

# 總用時
total_end = time.time()
print(f"總執行時間：{total_end - total_start:.2f} 秒")


擷取第 1 頁...
第 1 頁完成，用時 129.83 秒
擷取第 2 頁...
第 2 頁完成，用時 73.26 秒
所有資料擷取完畢
CSV 已儲存至：C:\Users\anaco\AppData\Local\Programs\Python\Python312\Scripts\coding output\台北市停車場.csv
總執行時間：207.46 秒


In [33]:
df

,停車場名稱,費率,地址,機車,小車,大車,電話,開放時間
0,大業路527巷平面停車場,小型車計次50元，機車免費。,臺北市北投區大業路527巷近捷運復興崗站,10,14,,,9-17
1,平陽街18號機車平面停車場,計時10元，當日當次最高上限30元(隔日另計)。,臺北市大同區平陽街18號旁,23,,,,9-20
2,中華路二段409巷平面停車場,計時30元,臺北市中正區中華路2段409巷1號對面,,5,,,9-17
3,南港公園平面停車場,計時20元,臺北市南港區東新街170號(南港公園大門旁),,84,,29441489,24小時
4,洲美公園臨時平面停車場,小型車計時20元，機車免費,臺北市北投區福美路199號旁,17,7,,,9-17
...,...,...,...,...,...,...,...,...
331,景豐臨時平面停車場,計時20元,臺北市文山區景豐街79號旁,,44,,89193669,24小時
332,華中橋堤外(一)平面停車場,小型車計時20元，機車免費,臺北市萬華區華中河濱公園網球場旁,19,71,,,9-17
333,洲子立體停車場,1.週一至週五(8-20)計時40元；(20-8)計時10元，夜間時段當次最高收費上限50元...,臺北市內湖區瑞光路513巷22弄2號,,382,,8751-1380,24小時
334,安祥公園旁平面停車場,計時30元,臺北市大安區信義路3段157巷10弄2號旁,,11,,.,一般小型車7-20，電動車充電格位24小時收費
